# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import sys
import pyrootutils
from pathlib import Path

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# Parameters

In [12]:
N = 10
MAX_RECORDINGS = 200
CLIPS_PER_SPECIES = 2000
MIN_XC_RECORDINGS = 100
BIRDNET_THRESHOLD = 0.92

# Select species

In [ ]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
    select_diff_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS,dl_more = True)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

Species:   0%|          | 0/11167 [00:00<?, ?it/s]

In [14]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
       genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order
       Vireo         15                 1                  1                 1
Thamnophilus         14                 1                  1                 1
    Calidris         11                 1                  1                 1
       Strix         11                 1                  1                 1
      Trogon         11                 1                  1                 1
  Synallaxis         11                 1                  1                 1
      Tringa         10                 1                  1                 1
   Empidonax         10                 1                  1                 1

Potential families (need >=N distinct genera):
                                     family  n_species  n_distinct_genus  n_distinct_family  n_distinct_order
            Tyrannidae (Tyrant Flycatchers)        119                53                  1   

In [15]:
TARGET_GENUS  = "Turdus"
TARGET_FAMILY = "Turdidae"
TARGET_ORDER  = "PASSERIFORMES"

In [16]:

diff_species = select_same_genus(TARGET_GENUS, N, pool)
diff_genus   = select_same_family(TARGET_FAMILY, N, pool)
diff_family  = select_same_order(TARGET_ORDER, N, pool)
diff_order   = select_diff_order(N, pool)

collections: dict[str, list[str]] = {
    "diff_species": [s.scientific_name for s in diff_species],
    "diff_genus":   [s.scientific_name for s in diff_genus],
    "diff_family":  [s.scientific_name for s in diff_family],
    "diff_order":   [s.scientific_name for s in diff_order],
}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for n in names:
        print(f"  {n}")

ValueError: Only 0 species found for genus Turdus, need 10

# Download from Xeno-canto

In [ ]:
from building.download import download_species_list

all_species = list({name for names in collections.values() for name in names})
print(f"Downloading {len(all_species)} unique species ({MAX_RECORDINGS} recordings each)…")

downloaded = download_species_list(all_species, max_recordings=MAX_RECORDINGS)

print("\nDownload summary:")
for name, paths in downloaded.items():
    print(f"  {name}: {len(paths)} files")

# BirdNET validation + dataset assembly

In [ ]:
from building.dataset import validate_and_build_all

dataset_paths = validate_and_build_all(
    collections,
    clips_per_species=CLIPS_PER_SPECIES,
    threshold=BIRDNET_THRESHOLD,
)

print("\nDatasets ready:")
for name, path in dataset_paths.items():
    print(f"  {name}: {path}")

# Sanity check

In [ ]:
from pathlib import Path

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips  {species_counts}")